# Persona Multi-LoRA workshop: compare and serve response adapters

**Primary question:** Does the selected client adapter improve held-out response behavior over the same frozen base and persona prompt?

**Time:** 25 minutes plus local inference.

Prerequisite: train the selected client in [05: train persona adapter](05_train_persona_adapter.ipynb).


## Evaluation contract

- Compare **base** and **adapted** modes on the exact same ordered IDs.
- Both modes receive the same persona profile and incoming email prompt; only LoRA weights differ.
- Use validation for workflow checks and qualitative prompt review. Run final test only after settings are frozen.
- Resume checkpoints only when backend, adapter, split IDs, and generation settings match.
- Token F1 and ROUGE-L are reference-overlap diagnostics, not measures of personality or factual quality.
- A human reviewer must assess voice, helpfulness, unsupported commitments, and boundary adherence.


In [ ]:
from __future__ import annotations

import os
import shlex
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

import email_workshop
from email_workshop.model_profiles import (
    DEFAULT_MODEL_PROFILE,
    model_profile_keys,
    resolve_model_profile,
)
from email_workshop.persona import (
    PersonaBackend,
    evaluate_persona,
    installed_persona_adapters,
    load_persona_dataset,
    load_persona_registry,
    persona_vllm_command,
    score_persona,
    verify_persona_adapter,
)

WORKSHOP_ROOT = Path(email_workshop.__file__).resolve().parents[1]
PERSONA_REGISTRY = load_persona_registry(WORKSHOP_ROOT)
CLIENT_ID = 'maya-chen-ceo'
INPUT_FILE = WORKSHOP_ROOT / PERSONA_REGISTRY[CLIENT_ID]['data_file']
PERSONA_RUN_ROOT = WORKSHOP_ROOT / Path(os.getenv('EMAIL_PERSONA_RUN', 'artifacts/persona-lora-run-2026'))
MODEL_CHOICE = DEFAULT_MODEL_PROFILE
MODEL = resolve_model_profile(MODEL_CHOICE)
DEVICE = 'cuda'
RUN_EVALUATION = False
RUN_FINAL_TEST = False
SHOW_RESPONSE_SAMPLES = False
RUN_VLLM_COMMAND = False
ALLOW_SMOKE = False
MAX_EVALUATION_ROWS = None
print(f'Kernel: {sys.executable}')
print(f"Client/adapter: {CLIENT_ID} -> {PERSONA_REGISTRY[CLIENT_ID]['adapter']}")
print(f'Persona run: {PERSONA_RUN_ROOT}')
print(f'Model choices: {model_profile_keys()}')
print(f'Available model choices: {model_profile_keys()}')


## Load one shared evaluation set

The reviewed CSV already contains stable train, validation, and test assignments. This notebook never resplits the data. Validation and test IDs are recorded in the adapter data manifest and checked before inference.


In [ ]:
splits, persona, profile_text, data_manifest = load_persona_dataset(
    WORKSHOP_ROOT,
    CLIENT_ID,
    INPUT_FILE,
)
manifest_path = PERSONA_RUN_ROOT / 'persona_adapters' / persona['adapter'] / 'manifest.json'
manifest = None
if manifest_path.exists():
    manifest = verify_persona_adapter(PERSONA_RUN_ROOT, WORKSHOP_ROOT, CLIENT_ID)
    if manifest['config']['model_id'] != MODEL.model_id or manifest['config']['model_revision'] != MODEL.revision:
        raise ValueError('MODEL_CHOICE does not match the trained adapter base revision.')
    if manifest['purpose'] == 'smoke' and not ALLOW_SMOKE:
        raise ValueError('Smoke adapters are pipeline checks only; set ALLOW_SMOKE = True explicitly.')
elif RUN_EVALUATION or RUN_FINAL_TEST:
    raise FileNotFoundError(f'Train the selected persona adapter first: {manifest_path}')
display(pd.Series({
    'client': persona['name'],
    'adapter': persona['adapter'],
    'validation_rows': len(splits['validation']),
    'test_rows_sealed': len(splits['test']),
    'adapter_purpose': manifest['purpose'] if manifest else 'not trained',
}))


## Load the selected adapter only when inference is enabled

The local backend loads one frozen base and the selected PEFT adapter. `base` mode temporarily disables LoRA; `adapted` mode activates it. Greedy decoding keeps the comparison deterministic.


In [ ]:
backend = None
if RUN_EVALUATION or RUN_FINAL_TEST:
    backend = PersonaBackend(PERSONA_RUN_ROOT, WORKSHOP_ROOT, CLIENT_ID, device=DEVICE)
    display(pd.Series({
        'backend': backend.identity['backend'],
        'device': backend.identity['device'],
        'client_id': backend.identity['client_id'],
        'max_new_tokens': backend.identity['generation']['max_new_tokens'],
    }))
else:
    print('Model loading skipped; enable RUN_EVALUATION or RUN_FINAL_TEST explicitly.')


## Compare base and adapted modes on validation

Use this stage to verify the pipeline and develop a fixed human-review rubric. Do not claim quality from a smoke run or from lexical overlap alone.


In [ ]:
validation_results = {}
if RUN_EVALUATION:
    validation_rows = splits['validation'][:MAX_EVALUATION_ROWS]
    for mode in ('base', 'adapted'):
        validation_results[mode] = evaluate_persona(
            validation_rows,
            backend,
            mode,
            PERSONA_RUN_ROOT / 'persona_evaluation' / persona['adapter'] / f'validation-{mode}.json',
            'validation',
        )
    if [row['id'] for row in validation_results['base']] != [row['id'] for row in validation_results['adapted']]:
        raise RuntimeError('Base and adapted evaluations used different IDs.')
else:
    print('Validation inference skipped; set RUN_EVALUATION = True to enable.')


## Interpret automatic metrics conservatively

- **Non-empty rate** catches generation failures.
- **Token F1** and **ROUGE-L** measure overlap with one reviewed reference reply.
- **Length ratio** detects extreme verbosity or truncation.
- **Exact match** is usually low for an open-ended writing task.

None of these determine whether the reply sounds like the person. Before final test, freeze a human rubric covering persona voice, relevance, actionable next step, unsupported commitments, confidentiality, and profile boundaries.


In [ ]:
if validation_results:
    validation_scores = pd.DataFrame({
        mode: score_persona(rows) for mode, rows in validation_results.items()
    }).T
    display(validation_scores)
else:
    print('No validation predictions yet.')


## Optional response-only qualitative review

This cell never prints the incoming email. It shows reference and generated replies only. Treat these as potentially sensitive and enable the gate only in an approved environment.


In [ ]:
if SHOW_RESPONSE_SAMPLES:
    if not validation_results:
        raise RuntimeError('Run validation inference before showing samples.')
    rows = []
    for index in range(min(5, len(validation_results['base']))):
        rows.append({
            'id': validation_results['base'][index]['id'],
            'reference': validation_results['base'][index]['expected'],
            'base': validation_results['base'][index]['generated'],
            'adapted': validation_results['adapted'][index]['generated'],
        })
    display(pd.DataFrame(rows))
else:
    print('Response sample display disabled.')


## Final test gate

Run this once after freezing model choice, adapter checkpoint, generation settings, and human rubric. Never use test outcomes to revise those choices in the same experiment.


In [ ]:
test_results = {}
if RUN_FINAL_TEST:
    test_rows = splits['test'][:MAX_EVALUATION_ROWS]
    for mode in ('base', 'adapted'):
        test_results[mode] = evaluate_persona(
            test_rows,
            backend,
            mode,
            PERSONA_RUN_ROOT / 'persona_evaluation' / persona['adapter'] / f'test-{mode}.json',
            'test',
        )
    if [row['id'] for row in test_results['base']] != [row['id'] for row in test_results['adapted']]:
        raise RuntimeError('Base and adapted test evaluations used different IDs.')
    display(pd.DataFrame({mode: score_persona(rows) for mode, rows in test_results.items()}).T)
else:
    print('Final test remains sealed; set RUN_FINAL_TEST = True only after freezing settings.')


## Inventory the shared Multi-LoRA run

Repeat notebooks 04–05 for each registered client using the same run root and model configuration. The inventory rejects mixed base revisions or hyperparameters because those adapters are not a valid shared Multi-LoRA comparison.


In [ ]:
installed = installed_persona_adapters(PERSONA_RUN_ROOT, WORKSHOP_ROOT)
if installed:
    display(pd.DataFrame([
        {
            'client_id': client_id,
            'adapter': item['adapter'],
            'purpose': item['purpose'],
            'train_rows': item['train_rows'],
            'model_id': item['config']['model_id'],
        }
        for client_id, item in installed.items()
    ]).set_index('client_id'))
else:
    print('No persona adapters installed in this run yet.')


## Optional local vLLM command

After multiple full adapters are trained, the helper builds one static vLLM command that registers every verified adapter against the shared base. Run it only in the separate Linux/WSL2 serving environment documented by the workshop. Runtime loading of untrusted adapters is intentionally not enabled.


In [ ]:
if RUN_VLLM_COMMAND:
    command = persona_vllm_command(PERSONA_RUN_ROOT, WORKSHOP_ROOT, 'artifacts/persona-lora-run-2026')
    print(shlex.join(command))
else:
    print('vLLM command generation skipped; set RUN_VLLM_COMMAND = True after training adapters.')


## Interpretation checklist

- Did adapted outputs improve the frozen human rubric, not only lexical overlap?
- Were base and adapted runs evaluated on identical IDs with identical prompts?
- Did any output make an unsupported commitment or violate the persona boundaries?
- Is the result from a full adapter rather than a smoke test?
- Were test results viewed only after settings were frozen?
- Can every served adapter be traced to the same base revision and training configuration?
